# RP-009 — Template Diversity Control (Colab via VS Code)

**Experiment:** FSC147 train subset 1000 × 10 epochs, `template-diversity` mode, class-name inference.

**Source:** `rich-prompt-alignment` at commit `d9601df91bfefa58f823eaf021cca617b6394ac7`.

**Precondition:** One-batch GPU smoke has **PASS** and no optimizer steps; the report is checked explicitly before training.

**Execution:** Run the setup cells in order. Full training is guarded by `RUN_FULL_TRAINING=False` by default. Change this single value to `True` only when you intend to launch the full run. Do **not** rerun the old RP-008 preflight/smoke cell, which would hit existing output guards.

**Data:** All large assets and outputs remain in the same Colab/Drive locations as RP-008. No Gemma API calls are needed.


In [10]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Verify source revision

In [11]:
from pathlib import Path
import os
import subprocess

REPO_DIR = Path('/content/T2ICount-Implementation')
REPO_URL = 'https://github.com/bqa100507-spec/T2ICount-Implementation.git'
BRANCH = 'rich-prompt-alignment'
EXPECTED_COMMIT = 'd9601df91bfefa58f823eaf021cca617b6394ac7'

if (REPO_DIR / '.git').is_dir():
    # Preserve tracked edits. Never reset/discard local code.
    subprocess.run(['git', 'diff', '--exit-code', 'HEAD', '--', '.'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'], cwd=REPO_DIR, check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f'Path exists but is not a Git repository: {REPO_DIR}')
else:
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
def git(*args):
    return subprocess.check_output(['git', '-C', str(REPO_DIR), *args], text=True).strip()

head = git('rev-parse', 'HEAD')
branch = git('branch', '--show-current')
remote = git('ls-remote', 'origin', 'refs/heads/' + BRANCH).split()[0]
assert branch == BRANCH and head == EXPECTED_COMMIT and remote == EXPECTED_COMMIT, (branch, head, remote)
assert not git('status', '--porcelain', '--untracked-files=no'), 'Review tracked local edits first.'
os.environ['RP009_EXPECTED_COMMIT'] = EXPECTED_COMMIT
print('Branch:', branch, 'Commit:', head, 'Remote:', remote)


Branch: rich-prompt-alignment Commit: d9601df91bfefa58f823eaf021cca617b6394ac7 Remote: d9601df91bfefa58f823eaf021cca617b6394ac7


## 2. Bootstrap local Colab assets

In [12]:
import os
import shutil
import subprocess
import time
import zipfile
from pathlib import Path

from tqdm.auto import tqdm

ASSET_ROOT = "/content/T2ICount-assets"
DRIVE_ASSET_ROOT = "/content/drive/MyDrive/T2ICount-assets"

os.environ["T2ICOUNT_ASSET_ROOT"] = ASSET_ROOT
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# Reuse the baseline notebook's Drive ZIP -> local SSD asset bootstrap.
DRIVE_ZIP = Path("/content/drive/MyDrive/T2ICount-assets.zip")
LOCAL_ZIP = Path("/content/T2ICount-assets.zip")
ASSET_ROOT_PATH = Path(ASSET_ROOT)
DRIVE_DIAGNOSTIC_ROOT = (
    Path(DRIVE_ASSET_ROOT) / "checkpoints/rich_prompt/diagnostic"
)
DRIVE_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)

REQUIRED_ASSETS = {
    "CLIP": (
        ASSET_ROOT_PATH / "pretrained/clip-vit-large-patch14",
        "directory",
    ),
    "Stable Diffusion": (
        ASSET_ROOT_PATH / "pretrained/sd-v1-5/v1-5-pruned-emaonly.ckpt",
        "file",
    ),
    "FSC147": (ASSET_ROOT_PATH / "datasets/FSC147", "directory"),
    "T2ICount checkpoint": (
        ASSET_ROOT_PATH / "checkpoints/official/best_model_paper.pth",
        "file",
    ),
}


def check_assets(verbose=True):
    status = {}
    for name, (path, kind) in REQUIRED_ASSETS.items():
        status[name] = path.is_dir() if kind == "directory" else path.is_file()

    if verbose:
        print("\nAsset status:")
        for name, ok in status.items():
            print(f"  [{'OK' if ok else 'MISSING'}] {name}")
            if not ok:
                print(f"       {REQUIRED_ASSETS[name][0]}")

    return all(status.values())


def human_size(num_bytes):
    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(num_bytes)
    for unit in units:
        if size < 1024:
            return f"{size:.2f} {unit}"
        size /= 1024
    return f"{size:.2f} PB"


def copy_with_progress(src, dst, chunk_size=64 * 1024 * 1024):
    total = src.stat().st_size
    dst.parent.mkdir(parents=True, exist_ok=True)
    with src.open("rb") as fsrc, dst.open("wb") as fdst:
        with tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc="Copy ZIP Drive -> local",
        ) as progress:
            while True:
                chunk = fsrc.read(chunk_size)
                if not chunk:
                    break
                fdst.write(chunk)
                progress.update(len(chunk))


if check_assets():
    print("\nLocal assets already complete:", ASSET_ROOT_PATH)
else:
    if not DRIVE_ZIP.is_file():
        raise FileNotFoundError(
            "Cannot find /content/drive/MyDrive/T2ICount-assets.zip. "
            "Upload the baseline asset ZIP to My Drive first."
        )

    drive_zip_size = DRIVE_ZIP.stat().st_size
    print("\nDrive ZIP found.")
    print("ZIP size:", human_size(drive_zip_size))

    local_zip_valid = (
        LOCAL_ZIP.is_file() and LOCAL_ZIP.stat().st_size == drive_zip_size
    )

    if local_zip_valid:
        print("\nLocal ZIP already exists with matching size.")
        print("Skipping Drive copy:", LOCAL_ZIP)
    else:
        # Remove only an incomplete local ZIP, never anything on Drive.
        if LOCAL_ZIP.exists():
            print("\nRemoving incomplete local ZIP:", LOCAL_ZIP)
            LOCAL_ZIP.unlink()

        free_space = shutil.disk_usage("/content").free
        print("Free local disk:", human_size(free_space))
        if free_space < drive_zip_size + 5 * 1024**3:
            raise RuntimeError("Not enough local disk space to safely copy the asset archive.")

        print("\nCopying ZIP from Google Drive to Colab SSD...")
        start = time.perf_counter()
        copy_with_progress(DRIVE_ZIP, LOCAL_ZIP)
        elapsed = time.perf_counter() - start
        print(f"\nZIP copied in {elapsed / 60:.1f} minutes.")

    print("\nChecking ZIP integrity...")
    if not zipfile.is_zipfile(LOCAL_ZIP):
        raise RuntimeError(f"Local archive is not a valid ZIP: {LOCAL_ZIP}")

    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        members = zf.infolist()
        names = [info.filename.replace('\\', '/') for info in members]
        has_root_folder = any(
            name.startswith("T2ICount-assets/") for name in names
        )
        total_uncompressed = sum(info.file_size for info in members)

    print("Uncompressed size:", human_size(total_uncompressed))
    print("Files in archive:", len(members))

    if has_root_folder:
        extract_to = Path("/content")
    else:
        ASSET_ROOT_PATH.mkdir(parents=True, exist_ok=True)
        extract_to = ASSET_ROOT_PATH

    print(
        "\nArchive layout:",
        "contains T2ICount-assets/"
        if has_root_folder
        else "contents start inside asset root",
    )
    print("Extract target:", extract_to)
    print("\nExtracting on Colab local SSD...")

    start = time.perf_counter()
    result = subprocess.run(
        ["unzip", "-q", "-o", str(LOCAL_ZIP), "-d", str(extract_to)],
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"unzip failed with return code {result.returncode}")

    elapsed = time.perf_counter() - start
    print(f"Extraction finished in {elapsed / 60:.1f} minutes.")
    print("\nValidating extracted assets...")
    if not check_assets():
        raise RuntimeError(
            "Extraction finished but required assets are still missing."
        )

    if LOCAL_ZIP.exists():
        print("\nRemoving local ZIP to recover disk space...")
        LOCAL_ZIP.unlink()

print("\n========================================")
print("LOCAL ASSET BOOTSTRAP PASSED")
print("========================================")
print("Runtime assets :", ASSET_ROOT_PATH)
print("Drive saves    :", DRIVE_DIAGNOSTIC_ROOT)
print("T2ICOUNT_ASSET_ROOT =", os.environ["T2ICOUNT_ASSET_ROOT"])


Asset status:
  [OK] CLIP
  [OK] Stable Diffusion
  [OK] FSC147
  [OK] T2ICount checkpoint

Local assets already complete: /content/T2ICount-assets

LOCAL ASSET BOOTSTRAP PASSED
Runtime assets : /content/T2ICount-assets
Drive saves    : /content/drive/MyDrive/T2ICount-assets/checkpoints/rich_prompt/diagnostic
T2ICOUNT_ASSET_ROOT = /content/T2ICount-assets


## 3. Install dependencies

In [13]:
%cd /content/T2ICount-Implementation
%pip install -q -r requirements-colab.txt


/content
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for taming-transformers (pyproject.toml) ... done
  Building editable for clip (pyproject.toml) ... done


## 4. Pinned `taming` source and GPU check

In [14]:
from pathlib import Path
import os
import subprocess
import sys

TAMING_DIR = Path('/content/taming-transformers')
TAMING_COMMIT = '3ba01b241669f5ade541ce990f7650a3b8f65318'
if not TAMING_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/CompVis/taming-transformers.git', str(TAMING_DIR)], check=True)
assert (TAMING_DIR / '.git').is_dir()
subprocess.run(['git', '-C', str(TAMING_DIR), 'diff', '--exit-code', 'HEAD', '--', '.'], check=True)
subprocess.run(['git', '-C', str(TAMING_DIR), 'checkout', '--detach', TAMING_COMMIT], check=True)
actual = subprocess.check_output(['git', '-C', str(TAMING_DIR), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == TAMING_COMMIT, actual
sys.path.insert(0, str(TAMING_DIR))
os.environ['PYTHONPATH'] = str(TAMING_DIR) + os.pathsep + os.environ.get('PYTHONPATH', '')
from taming.modules.vqvae.quantize import VectorQuantizer2
subprocess.run([sys.executable, '-c', 'from taming.modules.vqvae.quantize import VectorQuantizer2; print("taming subprocess: OK")'], check=True)

import torch
assert torch.cuda.is_available(), 'Colab GPU is required.'
print('GPU:', torch.cuda.get_device_name())
print('PyTorch:', torch.__version__)
print('taming source:', actual)


GPU: NVIDIA L4
PyTorch: 2.11.0+cu130
taming source: 3ba01b241669f5ade541ce990f7650a3b8f65318


## 5. Validate original prompt bank

In [15]:
from pathlib import Path

PROMPT_BANK = Path(
    "/content/drive/MyDrive/T2ICount-assets/prompts/"
    "fsc147_train1000_seed3407_gemma4_v3.json"
)
if not PROMPT_BANK.is_file():
    raise FileNotFoundError(
        f"Upload/copy fsc147_train1000_seed3407_gemma4_v3.json to {PROMPT_BANK.parent}"
    )

# Expected:
# selected_sample_count=1000
# selected_image_fingerprint=sha256:dd96b36bf15013e194b1a8ece06452a19822aae028fc00c0de019cbb7a311f24
!python tools/validate_rich_prompt_training_bank.py \
  --rich-prompt-bank /content/drive/MyDrive/T2ICount-assets/prompts/fsc147_train1000_seed3407_gemma4_v3.json \
  --data-dir /content/T2ICount-assets/datasets/FSC147 \
  --train-samples 1000 \
  --train-subset-seed 3407

Rich prompt bank validation passed.
selected_sample_count=1000
selected_image_fingerprint=sha256:dd96b36bf15013e194b1a8ece06452a19822aae028fc00c0de019cbb7a311f24
prompt_bank_fingerprint=sha256:c3ec587b37fd9f351cdedbce888b26e0ba7a020de956c4ef18202a1268e7e32f
protocol_version=rich-prompt-phase1-v3


## 6. Verify already completed RP-009 GPU smoke report

In [16]:
import json
from pathlib import Path

smoke_path = Path('/content/drive/MyDrive/T2ICount-assets/outputs/rp009/smoke_template_diversity_v1.json')
if not smoke_path.is_file():
    raise FileNotFoundError(f'RP-009 smoke report not found: {smoke_path}')
smoke = json.loads(smoke_path.read_text(encoding='utf-8'))
assert smoke['status'] == 'PASS' and smoke['optimizer_steps'] == 0 and smoke['backward_completed'] is True
assert smoke['git_commit'] == EXPECTED_COMMIT
assert smoke['initial_parameter_fingerprint'] == smoke['final_parameter_fingerprint']
assert smoke['rich_prompt_config']['rich_supervision_mode'] == 'template-diversity'
assert smoke['rich_prompt_config']['template_scheme_version'] == 'class-name-templates-v1'
assert smoke['rich_prompt_config']['train_samples'] == 1000
assert smoke['rich_prompt_config']['train_subset_seed'] == 3407
assert [item['caption'] for item in smoke['forwards']] == [
    'geese', 'a photo of geese', 'the geese in the image'
]
print('RP-009 GPU smoke: PASS; no optimizer update')
print('GPU:', smoke['gpu'], '| loss:', smoke['loss'], '| gradient L2:', smoke['gradients']['gradient_l2'])
print('Smoke report:', smoke_path)


RP-009 GPU smoke: PASS; no optimizer update
GPU: NVIDIA L4 | loss: 0.48266634345054626 | gradient L2: 1.7751418467415125
Smoke report: /content/drive/MyDrive/T2ICount-assets/outputs/rp009/smoke_template_diversity_v1.json


## 7. Full RP-009 training (explicit opt-in)

In [17]:
# Explicit launch gate. Leave False if you only want to verify the environment.
RUN_FULL_TRAINING = True

import os
os.environ['RP009_LAUNCH_TRAIN'] = '1' if RUN_FULL_TRAINING else '0'
print('RP-009 full training:', 'ENABLED' if RUN_FULL_TRAINING else 'DISABLED (safe default)')


RP-009 full training: ENABLED


In [18]:
%%bash
set -euo pipefail
if [ "${RP009_LAUNCH_TRAIN:-0}" != 1 ]; then
  echo 'RP-009 training skipped. Set RUN_FULL_TRAINING=True in the preceding cell to launch.'
  exit 0
fi
cd /content/T2ICount-Implementation
EXPECTED_COMMIT=d9601df91bfefa58f823eaf021cca617b6394ac7

test "$(git branch --show-current)" = rich-prompt-alignment
test "$(git rev-parse HEAD)" = "$EXPECTED_COMMIT"
test "$(git ls-remote origin refs/heads/rich-prompt-alignment | cut -f1)" = "$EXPECTED_COMMIT"
git diff --exit-code HEAD -- .
test "$(git -C /content/taming-transformers rev-parse HEAD)" = 3ba01b241669f5ade541ce990f7650a3b8f65318
export PYTHONPATH="/content/taming-transformers:${PYTHONPATH:-}"
python -c 'from taming.modules.vqvae.quantize import VectorQuantizer2; print("taming import: OK")'
export HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1
unset SSLKEYLOGFILE

# Safety: refuse existing full-run directories; never overwrite prior artifacts.
RUN_NAME=template_diversity_1000x10_v1
SAVE_ROOT=/content/drive/MyDrive/T2ICount-assets/checkpoints/rich_prompt/rp009
OUTPUT_DIR=/content/drive/MyDrive/T2ICount-assets/outputs/rp009/$RUN_NAME
test ! -e "$SAVE_ROOT/$RUN_NAME"
test ! -e "$OUTPUT_DIR"
mkdir -p "$OUTPUT_DIR"

python -u train.py \
  --asset-root /content/T2ICount-assets \
  --config configs/v1-inference.yaml \
  --save-dir "$SAVE_ROOT" --content "$RUN_NAME" \
  --rich-prompt-bank /content/drive/MyDrive/T2ICount-assets/prompts/fsc147_train1000_seed3407_gemma4_v3.json \
  --rich-supervision-mode template-diversity --rich-consistency-weight 0 \
  --seed 3407 --train-samples 1000 --train-subset-seed 3407 \
  --batch-size 1 --epochs 10 --start-epoch 0 --start-val 7 --val-epoch 1 \
  --num-workers 0 --crop-size 384 --concat-size 224 --downsample-ratio 8 \
  --stride 384 --lr 5e-5 --weight-decay 5e-4 --checkpoint-interval 5 \
  2>&1 | tee "$OUTPUT_DIR/console.log"


taming import: OK
10-27 11:42:52, asset_root:/content/T2ICount-assets
10-27 11:42:52, content:template_diversity_1000x10_v1
10-27 11:42:52, seed:3407
10-27 11:42:52, crop_size:384
10-27 11:42:52, concat_size:224
10-27 11:42:52, downsample_ratio:8
10-27 11:42:52, data_dir:/content/T2ICount-assets/datasets/FSC147
10-27 11:42:52, config:/content/T2ICount-Implementation/configs/v1-inference.yaml
10-27 11:42:52, sd_path:/content/T2ICount-assets/pretrained/sd-v1-5/v1-5-pruned-emaonly.ckpt
10-27 11:42:52, clip_path:/content/T2ICount-assets/pretrained/clip-vit-large-patch14
10-27 11:42:52, save_dir:/content/drive/MyDrive/T2ICount-assets/checkpoints/rich_prompt/rp009
10-27 11:42:52, max_num:2
10-27 11:42:52, checkpoint_interval:5
10-27 11:42:52, resume:
10-27 11:42:52, batch_size:1
10-27 11:42:52, smoke_train_samples:0
10-27 11:42:52, train_samples:1000
10-27 11:42:52, train_subset_seed:3407
10-27 11:42:52, rich_prompt_bank:/content/drive/MyDrive/T2ICount-assets/prompts/fsc147_train1000_seed340

## After training

Do **not** interpret training loss as the research conclusion. Review the validation-selected `best_model_*.pth` and run the original class-name FSC147 test evaluation, then compare MAE/RMSE against BASE-002 and RP-003. Preserve training logs, checkpoint files and source/runtime provenance.

For a rerun, choose a **new** `RUN_NAME`, do not delete or overwrite prior outputs.
